In [ ]:
import pyxnat
import os
import datetime
import pandas as pd
import re
import os
import glob
import itertools 
import joblib
from enum import Enum
from pathlib import Path
import sys

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import pymcm

from nilearn import image, plotting, masking

sys.path.append('/RAID1/jupytertmp/mcm/src')
import filenames

%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

In [ ]:
def test_file(file:str) -> bool:
    return os.path.exists(file)

---

# Setup

In [ ]:
subjects = [3, 7, 12, 14, 17, 20, 23, 25, 26, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38]

cohort2017 = subjects[:9]
cohort2021 = subjects[9:]

## Docker stuff

In [ ]:
container_name = 'cooltool'

container = ! docker ps | grep cooltool | awk '{print $1}'
if not container:

    container, = ! docker run \
        --name {container_name} \
        --gpus all \
        -it -d \
        -v /RAID1/jupytertmp/:/RAID1/jupytertmp \
        -v /RAID1/tmp:/RAID1/tmp\
        -v /RAID1/xnat:/RAID1/xnat:ro\
        valneurolab/supertool:v2
    
    print(f'started new container {container}')
else:
    container = container[0]
    print(f'acquired running container {container}')

container = container[:5]

## Prepare directories

Run this from the terminal
```bash
for sub in sub-s0*; do mkdir -p $sub/{cpac,degree-centrality,niftypet-recon}; done
```

---

# Copy or download data

### Download the bids resource

In [ ]:
for subject in subjects:

    zip_download = f'/RAID1/jupytertmp/mcm/data/tmp/s{subject:03}.zip'
    files = f'/RAID1/jupytertmp/mcm/data/tmp/s{subject:03}-AUF/resources/bids/files/*'
    dst = f'/RAID1/jupytertmp/mcm/data/bids/sub-s{subject:03}/'
    url = f'http://10.0.3.12/data/projects/fdgquant2016/subjects/s{subject:03}/experiments/s{subject:03}-AUF/resources/bids/files?format=zip'

    ! mkdir {dst} && \
    curl --netrc -X GET {url} --output {zip_download} && \
    unzip {zip_download} -d /RAID1/jupytertmp/mcm/data/tmp/ && \
    mv {files} {dst}

### Download the mriqc resource 

In [ ]:
for subject in subjects:

    zip_download = f'/RAID1/jupytertmp/mcm/data/tmp/s{subject:03}.zip'
    files = f'/RAID1/jupytertmp/mcm/data/tmp/s{subject:03}-AUF/resources/mriqc/files/*'
    dst = f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{subject:03}/mriqc'
    url = f'http://10.0.3.12/data/projects/fdgquant2016/subjects/s{subject:03}/experiments/s{subject:03}-AUF/resources/mriqc/files?format=zip'

    ! mkdir {dst} && \
    curl --netrc -X GET {url} --output {zip_download} && \
    unzip {zip_download} -d /RAID1/jupytertmp/mcm/data/tmp/ && \
    mv {files} {dst}

## FUNC

### fMRI timeseries

processed fMRI time series in functional space

In [ ]:
for subject in subjects:
    src = f'/RAID1/xnat/xnat-docker-compose/xnat-data/archive/fdgquant2016/arc001/s{subject:03}-AUF/RESOURCES/cpac_v1.4.0/functional_freq_filtered/_scan_rest/_compcor_ncomponents_5_selector_pc10.linear1.wm0.global0.motion1.quadratic1.gm0.compcor1.csf1/_bandpass_freqs_0.01.0.1/bandpassed_demeaned_filtered.nii.gz'
    dst = filenames.bold.format(id=subject)
    assert test_file(src)
    ! cp {src} {dst}

    src = f'/RAID1/xnat/xnat-docker-compose/xnat-data/archive/fdgquant2016/arc001/s{subject:03}-AUF/RESOURCES/cpac_v1.4.0/motion_correct/_scan_rest/sub-s{subject:03}_task-rest_bold_calc_tshift_resample_volreg.nii.gz'
    dst = filenames.bold_mc.format(id=subject)
    assert test_file(src)
    ! cp {src} {dst}

    # brain mask in func
    src = f'/RAID1/xnat/xnat-docker-compose/xnat-data/archive/fdgquant2016/arc001/s{subject:03}-AUF/RESOURCES/cpac_v1.4.0/functional_brain_mask/_scan_rest/sub-s{subject:03}_task-rest_bold_calc_tshift_resample_volreg_mask.nii.gz'
    dst = filenames.bold_brain_mask.format(id=subject)
    assert test_file(src)
    ! cp {src} {dst}

## Diffusion

In [ ]:
for subject in subjects:

    # Fiber orientation density
    src = f'/RAID1/xnat/xnat-docker-compose/xnat-data/archive/fdgquant2016/arc001/s{subject:03}-AUF/RESOURCES/mrtrix3_v0.4.2/dwi/sub-*{subject:03}_dwi_fod.mif'
    dst = filenames.fod.format(id=subject)
    # ! cp {src} {dst}

    # Denoised, bias-corrected dwi 4d
    src = f'/RAID1/xnat/xnat-docker-compose/xnat-data/archive/fdgquant2016/arc001/s{subject:03}-AUF/RESOURCES/mrtrix3_v0.4.2/dwi/sub-*{subject:03}_dwi_dn-preproc-bcor.mif'
    dst = filenames.dwi.format(id=subject)
    # ! cp {src} {dst}

    # Denoised, bias-corrected dwi 4d
    src = f'/RAID1/xnat/xnat-docker-compose/xnat-data/archive/fdgquant2016/arc001/s{subject:03}-AUF/RESOURCES/mrtrix3_v0.4.2/dwi/sub-*{subject:03}_dwi_dn-preproc-bcor-3D.nii.gz'
    dst = filenames.b0.format(id=subject)
    # ! cp {src} {dst}

    # 5tt segmentation
    src = f'/RAID1/xnat/xnat-docker-compose/xnat-data/archive/fdgquant2016/arc001/s{subject:03}-AUF/RESOURCES/mrtrix3_v0.4.2/anat/sub-*{subject:03}_T1w_fov_bet_regDWI_seg.nii.gz'
    dst = filenames.act_5tt.format(id=subject)
    ! cp {src} {dst}

    # Original tracts
    src = f'/RAID1/xnat/xnat-docker-compose/xnat-data/archive/fdgquant2016/arc001/s{subject:03}-AUF/RESOURCES/mrtrix3_v0.4.2/dwi/sub-*{subject:03}_tracts.tck'
    dst = filenames.tracts.format(id=subject)
    # ! cp {src} {dst}

    # Sifted tracts
    src = f'/RAID1/xnat/xnat-docker-compose/xnat-data/archive/fdgquant2016/arc001/s{subject:03}-AUF/RESOURCES/mrtrix3_v0.4.2/dwi/sub-*{subject:03}_tracts-sift.tck'
    dst = filenames.tracts_sift.format(id=subject)
    # ! cp {src} {dst}

    print(subject, end='\r')

## PET

### CMR Glc pet resampled to 3mm

In [ ]:
for subject in subjects:
    # src = f'/RAID1/jupytertmp/fdgquant2016/AUF/pet/s{subject:03}-AUF/niftypet_1120/PET/multiple-frames/trimmed/*trimmed-upsampled-scale-2_45min_mcf_fwhm-6_quant-cmrglc_acq-2242min_pvc-pveseg_mni-3mm.nii.gz'
    src = f'/RAID1/tmp/fdgquant2016/AUF/pet/s{subject:03}-AUF/niftypet_1120/PET/multiple-frames/trimmed/*2242*pvc-pveseg*pet*.nii.gz'
    assert len(glob.glob(src)) == 1
    dst = filenames.cmrglc_3mm.format(id=subject)
    ! cp {src} {dst}

### PET 32 frames recon

In [ ]:
for subject in subjects:
    src = f'/RAID1/tmp/fdgquant2016/AUF/pet/s{subject:03}-AUF/niftypet_1120/PET/multiple-frames/trimmed/*nfrm-3*trimmed-upsampled-scale-2_45min_mcf.nii.gz'
    if subject == 38:
        src = f'/RAID1/tmp/fdgquant2016/AUF/pet/s{subject:03}-AUF/niftypet_1120/PET/multiple-frames/trimmed/*nfrm-20*trimmed-upsampled-scale-2_45min_mcf.nii.gz'
    assert len(glob.glob(src)) == 1
    dst = filenames.pet.format(id=subject)
    ! cp {src} {dst}

    src = f'/RAID1/tmp/fdgquant2016/AUF/pet/s{subject:03}-AUF/niftypet_1120/PET/multiple-frames/trimmed/*.json'
    assert len(glob.glob(src)) == 1
    dst = filenames.recon_info.format(id=subject)
    ! cp {src} {dst}

### Arterial input function

In [ ]:
for subject in subjects:
    src = f'/RAID1/xnat/xnat-docker-compose/xnat-data/archive/fdgquant2016/arc001/s{subject:03}-AUF/RESOURCES/niftypet/*.crv'
    dst = filenames.aif.format(id=subject)
    ! cp {src} {dst}

## MNI templates

In [ ]:
! mkdir {rootdir}/mni

### MNI152

Copy from the contianer 

In [ ]:
! docker exec {container} cp /usr/local/fsl/data/standard/MNI152_T1_{{1,2}}mm.nii.gz {rootdir}/mni
! docker exec {container} cp /usr/local/fsl/data/standard/MNI152_T1_{{1,2}}mm_brain.nii.gz {rootdir}/mni

#### Resample 1mm to 3mm

In [ ]:
for template in glob.glob(f'{rootdir}/mni/MNI152_T1_1mm*'):
    template_3mm = template.replace('1mm', '3mm')
    # ! docker exec {container} 3dresample -input {template} -dxyz 3 3 3 -prefix {template_3mm}
    ! docker exec {container} flirt -in {template} -ref {template} -applyisoxfm 3.0 -nosearch -out {template_3mm}

### MNI ICBM152

Download from [here](https://www.bic.mni.mcgill.ca/ServicesAtlases/ICBM152NLin2009)

In [ ]:
url = "http://www.bic.mni.mcgill.ca/~vfonov/icbm/2009/mni_icbm152_nlin_asym_09a_nifti.zip"
! wget -O {Filenames.ROOTDIR}/mni/asym_09a.zip {url}
! unzip {Filenames.ROOTDIR}/mni/asym_09a.zip -d {Filenames.ROOTDIR}/mni && rm {Filenames.ROOTDIR}/mni/asym_09a.zip
! gzip {Filenames.ROOTDIR}/mni/mni_icbm152_nlin_asym_09a/*

In [ ]:
mni = '/RAID1/jupytertmp/mcm/mni/mni_icbm152_nlin_asym_09a/mni_icbm152_t1_tal_nlin_asym_09a.nii.gz'
mask = '/RAID1/jupytertmp/mcm/mni/mni_icbm152_nlin_asym_09a/mni_icbm152_t1_tal_nlin_asym_09a_mask.nii.gz'
out = '/RAID1/jupytertmp/mcm/mni/mni_icbm152_nlin_asym_09a/mni_icbm152_t1_tal_nlin_asym_09a_brain.nii.gz'
! docker exec {container} fslmaths {mni} -mul {mask} {out}

## Parcellations

### Yeo networks atlas

#### MNI space

In [ ]:
for nets, resolution in itertools.product(['7Networks', '17Networks'], ['1mm', '2mm']):
    
    url = f'https://github.com/ThomasYeoLab/CBIG/raw/master/stable_projects/brain_parcellation/Yeo2011_fcMRI_clustering/1000subjects_reference/Yeo_JNeurophysiol11_SplitLabels/MNI152/Yeo2011_{nets}_N1000.split_components.FSL_MNI152_{resolution}.nii.gz'
    ! wget -O {rootdir}/yeo/$(basename {url}) {url}
    
    lut_url = f'https://github.com/ThomasYeoLab/CBIG/raw/master/stable_projects/brain_parcellation/Yeo2011_fcMRI_clustering/1000subjects_reference/Yeo_JNeurophysiol11_SplitLabels/MNI152/{nets}_ColorLUT_fslview.lut'
    ! wget -O {rootdir}/yeo/$(basename {lut_url}) {lut_url}

#### Fsaverage5 surface

In [ ]:
for hemi, nets in itertools.product(['lh', 'rh'], ['7Networks', '17Networks']):
    
    url = f'https://github.com/ThomasYeoLab/CBIG/raw/master/stable_projects/brain_parcellation/Yeo2011_fcMRI_clustering/1000subjects_reference/Yeo_JNeurophysiol11_SplitLabels/fsaverage5/label/{hemi}.Yeo2011_{nets}_N1000.annot'
    ! wget -O {rootdir}/yeo/$(basename {url}) {url}

#### Hires fsaverage

In [ ]:
url = 'ftp://surfer.nmr.mgh.harvard.edu/pub/data/Yeo_JNeurophysiol11_FreeSurfer.zip'
! wget -O {rootdir}/yeo/yeo_hires.zip {url}
! unzip {rootdir}/yeo/yeo_hires.zip -d {rootdir}/yeo

### Schefer 400 roi atlas

Download from [Schaefer github](https://github.com/ThomasYeoLab/CBIG/tree/master/stable_projects/brain_parcellation/Schaefer2018_LocalGlobal) directory

#### MNI volume space

In [ ]:
for nrois, nets, resolution in itertools.product(['400Parcels'], ['7Networks', '17Networks'], ['1mm', '2mm']):

    url = f'https://github.com/ThomasYeoLab/CBIG/raw/master/stable_projects/brain_parcellation/Schaefer2018_LocalGlobal/Parcellations/MNI/Schaefer2018_{nrois}_{nets}_order_FSLMNI152_{resolution}.nii.gz'
    ! wget -O {rootdir}/schaefer/$(basename {url}) {url}
    
    lut_url = f'https://raw.githubusercontent.com/ThomasYeoLab/CBIG/master/stable_projects/brain_parcellation/Schaefer2018_LocalGlobal/Parcellations/MNI/fsleyes_lut/Schaefer2018_{nrois}_{nets}_order.lut'
    ! wget -O {rootdir}/schaefer/$(basename {lut_url}) {lut_url}

#### Resample to 3mm MNI

In [ ]:
mni3mm = f'{rootdir}/mni/MNI152_T1_3mm_brain.nii.gz'

for nets in ['7Networks', '17Networks']:
    
    atlas = f'{rootdir}/schaefer/Schaefer2018_400Parcels_{nets}_order_FSLMNI152_1mm.nii.gz'
    out = f'{rootdir}/schaefer/Schaefer2018_400Parcels_{nets}_order_FSLMNI152_3mm.nii.gz'
    ! docker exec {container} 3dresample -input {atlas} -master {mni3mm} -prefix {out} 

#### Fsaverage surface space

In [ ]:
! mkdir -p /RAID1/jupytertmp/mcm/schaefer/{fsaverage,fsaverage5}

In [ ]:
for resolution, hemi, nrois, nets in itertools.product(['fsaverage', 'fsaverage5'], ['lh', 'rh'], ['400Parcels'], ['7Networks', '17Networks']):
    
    url = f'https://github.com/ThomasYeoLab/CBIG/raw/master/stable_projects/brain_parcellation/Schaefer2018_LocalGlobal/Parcellations/FreeSurfer5.3/{resolution}/label/{hemi}.Schaefer2018_{nrois}_{nets}_order.annot'
    ! wget -O {rootdir}/schaefer/{resolution}/$(basename {url}) {url}

### Glasser atlas 

Dowload volume from [here](https://figshare.com/articles/dataset/HCP-MMP1_0_projected_on_MNI2009a_GM_volumetric_in_NIfTI_format/3501911)  
and surface from [here](https://figshare.com/articles/dataset/HCP-MMP1_0_projected_on_fsaverage/3498446)

#### MNI ICBM2009a nonlinear 

In [ ]:
! wget -O {rootdir}/hcp_mmp/glasser.zip https://figshare.com/ndownloader/articles/3501911/versions/5
! unzip {rootdir}/hcp_mmp/glasser.zip -d {rootdir}/hcp_mmp

#### Fix the floating point labels

- round to the nearest integer
- add 160 to the right heisphere
- and overwrite the original image

In [ ]:
glasser = nib.load(Filenames.GLASSER)
glasser_data = glasser.get_fdata()

glasser_int = np.rint(glasser_data).astype(np.uint16)

mid_idx = int(glasser.shape[0] / 2) + 1 # right hemisphere starts here
glasser_int[mid_idx:, ...] = np.where(glasser_int[mid_idx:, ...] > 0, glasser_int[mid_idx:, ...] + glasser_int.max(), glasser_int[mid_idx:, ...])

nib.save(nib.Nifti1Image(glasser_int, affine=glasser.affine, header=glasser.header), Filenames.GLASSER)

#### Fsaverage

In [ ]:
! wget -O {rootdir}/hcp_mmp/glasser_fsavg.zip https://figshare.com/ndownloader/articles/3498446/versions/2
! unzip {rootdir}/hcp_mmp/glasser_fsavg.zip -d {rootdir}/hcp_mmp

## BigBrain

The BigBrain data comes from the bigbrainwarp toolbox by Casey Paquola.  
- gihub: https://github.com/caseypaquola/BigBrainWarp
- paper: https://elifesciences.org/articles/70119  
- docs: https://bigbrainwarp.readthedocs.io/en/latest/index.html

In [1]:
! curl https://fz-juelich.sciebo.de/s/HoKucnh8zFpSbKF/download --output /RAID1/jupytertmp/mcm/data/external/bigbrain/BBW_BigData.zip && \
    unzip /RAID1/jupytertmp/mcm/data/external/bigbrain/BBW_BigData.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2813M    0 2813M    0     0  17.2M      0 --:--:--  0:02:43 --:--:-- 31.7M  0:00:24 --:--:-- 3516k:--:-- 21.1M-  0:02:30 --:--:-- 18.8M
unzip:  cannot find or open /RAID1/jupytertmp/mcm/bigbrain/BBW_BigData.zip, /RAID1/jupytertmp/mcm/bigbrain/BBW_BigData.zip.zip or /RAID1/jupytertmp/mcm/bigbrain/BBW_BigData.zip.ZIP.


In [ ]:
# convert layer thickness data to gii
bigbrain_template = "/RAID1/jupytertmp/mcm/bigbrain/BBW_BigData/spaces/tpl-bigbrain/tpl-bigbrain_hemi-{}_desc-Func_G1.shape.gii"

for hemi in ['L', 'R']:
    for layer in range(1,7):
        file = f'/RAID1/jupytertmp/mcm/bigbrain/BBW_BigData/spaces/tpl-bigbrain/tpl-bigbrain_hemi-{hemi}_desc-layer{layer}_thickness.txt'
        data = np.loadtxt(file)

        tpl = nib.load(bigbrain_template.format(hemi))
        tpl.darrays[0].data = np.float32(data)

        out = f'/RAID1/jupytertmp/mcm/bigbrain/layers/tpl-bigbrain_hemi-{hemi}_desc-layer{layer}.shape.gii'

        nib.save(tpl, out)

In [ ]:
# remap from bigbrain to fsaverage
for hemi in ['L', 'R']:
    msm_mesh = f"/RAID1/jupytertmp/mcm/bigbrain/BBW_BigData/xfms/tpl-fsaverage_hemi-{hemi}_den-164k_desc-sphere_rsled_like_bigbrain.reg.surf.gii"
    in_mesh= f'/RAID1/jupytertmp/mcm/bigbrain/BBW_BigData/spaces/tpl-fsaverage/tpl-fsaverage_hemi-{hemi}_den-164k_desc-sphere.surf.gii'
    for layer in range(1,7):
        in_gii = f'/RAID1/jupytertmp/mcm/bigbrain/layers/tpl-bigbrain_hemi-{hemi}_desc-layer{layer}.shape.gii'
        out = f'/RAID1/jupytertmp/mcm/bigbrain/layers/tpl-fsaverage_hemi-{hemi}_desc-layer{layer}.shape.gii'

        ! docker exec {container} wb_command -metric-resample \
            {in_gii} \
            {msm_mesh} \
            {in_mesh} \
            BARYCENTRIC \
            {out} \
        


In [ ]:
# convert schaefer from fsaverage to bigbrain

for hemi in ['L', 'R']:
    
    # annot to gifti
    annot = f'/RAID1/jupytertmp/mcm/schaefer/fsaverage/{hemi.lower()}h.Schaefer2018_400Parcels_7Networks_order.annot'
    ref = f'/RAID1/jupytertmp/mcm/bigbrain/spaces/tpl-fsaverage/tpl-fsaverage_hemi-{hemi}_den-164k_desc-aparc.label.gii'
    annot_gii = f'/RAID1/jupytertmp/mcm/schaefer/bigbrain/{hemi.lower()}h.Schaefer2018_400Parcels_7Networks_order.label.gii'

    values, _, _ = nib.freesurfer.io.read_annot(annot)
    ref_gifti = nib.load(ref)
    ref_gifti.darrays[0].data = np.float32(values)
    nib.save(ref_gifti, annot_gii)

    # remap to bigbrain
    msm_mesh = f'/RAID1/jupytertmp/mcm/bigbrain/xfms/tpl-bigbrain_hemi-{hemi}_desc-sphere_rsled_like_fsaverage.reg.surf.gii'
    in_mesh = f'/RAID1/jupytertmp/mcm/bigbrain/xfms/tpl-bigbrain_hemi-{hemi}_desc-sphere_rot_fsaverage.surf.gii'
    out = f'/RAID1/jupytertmp/mcm/schaefer/bigbrain/tpl-bigbrain_{hemi.lower()}h.Schaefer2018_400Parcels_7Networks_order.label.gii'
    
    ! docker exec {container} wb_command -label-resample \
        {annot_gii} \
		{msm_mesh} \
        {in_mesh} \
        BARYCENTRIC \
		{out}

## Synaptic density

In [5]:
url = 'https://xtra.nru.dk/SV2A-atlas/data/SV2A_atlas.zip'
dest = '/RAID1/jupytertmp/mcm/data/external'

! curl {url} --output {dest}/sva.zip && unzip {dest}/sva.zip -d {dest} && rm {dest}/sva.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 63.5M  100 63.5M    0     0  42.6M      0  0:00:01  0:00:01 --:--:-- 42.6M
Archive:  /RAID1/jupytertmp/mcm/data/external/sva.zip
   creating: /RAID1/jupytertmp/mcm/data/external/SV2A_atlas/
  inflating: /RAID1/jupytertmp/mcm/data/external/SV2A_atlas/Bmax.mean.fsaverage.lh.nii.gz  
  inflating: /RAID1/jupytertmp/mcm/data/external/SV2A_atlas/Bmax.mean.fsaverage.rh.nii.gz  
  inflating: /RAID1/jupytertmp/mcm/data/external/SV2A_atlas/Bmax.mean.MNI152.sm5.nii.gz  
  inflating: /RAID1/jupytertmp/mcm/data/external/SV2A_atlas/regional_values.csv  
  inflating: /RAID1/jupytertmp/mcm/data/external/SV2A_atlas/Bmax.cov.fsaverage.lh.nii.gz  
  inflating: /RAID1/jupytertmp/mcm/data/external/SV2A_atlas/Bmax.cov.fsaverage.rh.nii.gz  
  inflating: /RAID1/jupytertmp/mcm/data/external/SV2A_atlas/Bmax.cov.MNI152.sm5.nii.gz  
  inflating: /RAID1/j

## AHBA data

In [6]:
import abagen
abagen.fetch_rnaseq(data_dir='/RAID1/jupytertmp/mcm/data/external/AHBA')
abagen.fetch_microarray(data_dir='/RAID1/jupytertmp/mcm/data/external/AHBA')


Dataset created in /RAID1/jupytertmp/mcm/data/external/AHBA/microarray



Downloaded 166233851 of 166233851 bytes (100.0%,    0.0s remaining) ...done. (34 seconds, 0 min)
Extracting data from /RAID1/jupytertmp/mcm/data/external/AHBA/microarray/e6f4cbce4d5f941b7b805be2308e2067/normalized_microarray_donor12876/donor12876.zip..... done.


{'12876': {'microarray': '/RAID1/jupytertmp/mcm/data/external/AHBA/microarray/normalized_microarray_donor12876/MicroarrayExpression.csv',
  'ontology': '/RAID1/jupytertmp/mcm/data/external/AHBA/microarray/normalized_microarray_donor12876/Ontology.csv',
  'pacall': '/RAID1/jupytertmp/mcm/data/external/AHBA/microarray/normalized_microarray_donor12876/PACall.csv',
  'probes': '/RAID1/jupytertmp/mcm/data/external/AHBA/microarray/normalized_microarray_donor12876/Probes.csv',
  'annotation': '/RAID1/jupytertmp/mcm/data/external/AHBA/microarray/normalized_microarray_donor12876/SampleAnnot.csv'}}

---

# Preprocessing

## fMRI timeseries mean and std

before nuisance regression - for registration

In [ ]:
for subject in subjects:
    
    bold = filenames.bold_mc.format(id=subject)
    mean = filenames.bold_mean.format(id=subject)
    std =  filenames.bold_std.format(id=subject)
    snr =  filenames.bold_snr.format(id=subject)
    snr_mask =  filenames.bold_snr_mask.format(id=subject)
    brain_mask = filenames.bold_brain_mask.format(id=subject)

    ! docker exec {container} fslmaths {bold} -Tmean {mean}
    ! docker exec {container} fslmaths {bold} -Tstd {std}
    ! docker exec {container} fslmaths {mean} -div {std} {snr}
    ! docker exec {container} fslmaths {snr} -mas {brain_mask} -thrP 25 -bin {snr_mask}

    ! docker exec {container} fslstats {snr} -k {snr_mask} -r -R

## PET Preprocessing

### Mean pet

Take average only of last 5 frames to avoid noise, esp in the beginning 

In [ ]:
nframes = 5

for subject in subjects:

    pet = filenames.pet.format(id=subject)
    last5 = filenames.pet_last5.format(id=subject)
    mean = filenames.pet_mean.format(id=subject)

    _, nvols = ! docker exec {container} fslnvols {pet}
    start_frame = int(nvols) - nframes

    ! docker exec {container} fslroi {pet} {last5} {start_frame} {nframes}
    ! docker exec {container} fslmaths {last5} -Tmean {mean}


### Brain extraction 

In [ ]:
for subject in subjects:

    pet = filenames.pet_mean.format(id=subject)
    brain = filenames.pet_brain.format(id=subject)
    mask = filenames.pet_brainmask.format(id=subject)

    ! docker exec {container} mri_synthstrip -i {pet} -o {brain} -m {mask}

### Head mask for PET 

In [ ]:
for subject in subjects:

    print(subject, end='\r')
    
    pet = filenames.pet_mean.format(id=subject)
    out = filenames.pet_headmask.format(id=subject)

    ! docker exec {container} fslmaths \
        {pet} \
        -thrP 10 \
        -fmedian \
        -bin \
        -dilM \
        -fillh \
        {out} \
        -odt char

## Anatomical preprocessing

### antsCorticalThickness

In [ ]:
for subject in subjects:
    zip_download = f'/RAID1/jupytertmp/mcm/data/tmp/s{subject:03}.zip'
    files = f'/RAID1/jupytertmp/mcm/data/tmp/s{subject:03}-AUF/resources/ants-cortical-thickness/files/*'
    dst = str(filenames.ants).format(id=subject)
    url = f'http://10.0.3.12/data/projects/fdgquant2016/subjects/s{subject:03}/experiments/s{subject:03}-AUF/resources/ants-cortical-thickness/files?format=zip'

    ! mkdir {dst} && \
    curl --netrc -X GET {url} --output {zip_download} && \
    unzip {zip_download} -d /RAID1/jupytertmp/mcm/data/tmp/ && \
    mv {files} {dst}

In [ ]:
for subject in subjects:
    
    mask = f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{subject:03}/ants-cortical-thickness/*BrainExtractionMask.nii.gz'
    mask_bids = filenames.t1_brainmask.format(id=subject)
    ! mv {mask} {mask_bids}

    t1 = f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{subject:03}/ants-cortical-thickness/*BrainSegmentation0N4.nii.gz'
    t1_bids = filenames.t1.format(id=subject)
    ! mv {t1} {t1_bids}

    thickness = f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{subject:03}/ants-cortical-thickness/*CorticalThickness.nii.gz'
    thickness_bids = filenames.thickness.format(id=subject)
    ! mv {thickness} {thickness_bids}

    brain = filenames.t1_brain.format(id=subject)
    ! docker exec {container} fslmaths {t1_bids} -mas {mask_bids} {brain}

    segmentation = f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{subject:03}/ants-cortical-thickness/*BrainSegmentation.nii.gz'
    for i, label in zip(range(1,7), ['csf', 'gm', 'wm', 'sgm', 'bs', 'cbm']):

        probseg = f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{subject:03}/ants-cortical-thickness/*BrainSegmentationPosteriors{i}.nii.gz'
        probseg_bids = filenames.get(f"{label}_prob").format(id=subject)
        ! mv {probseg} {probseg_bids}

        dseg = filenames.get(f"{label}_mask").format(id=subject)
        ! docker exec {container} fslmaths {segmentation} -thr {i} -uthr {i} -bin {dseg}

### freesurfer

---

# Registration

**DO NOT MESS UP THE TEMPLATE HERE**  
↓↓↓ 

In [ ]:
MNI = Filenames.MNI152
ATLAS = Filenames.ATLAS

↑↑↑

## `anat -> mni`

the command takes 2.5 hours with 7 jobs, 5 cpus each  
```bash
cd /RAID1/jupytertmp/mcm/data/bids/derivatives/
parallel --jobs 5 --delay 60 --progress /RAID1/jupytertmp/mcm/src/launcher_anat2mni.sh ::: sub-s*  
slicesdir -p /RAID1/jupytertmp/mcm/data/external/mni/MNI152_T1_1mm_brain.nii.gz sub-s*/registration/*anat2mni*
```

## `pet -> anat`

```bash
parallel --jobs 10 --progress /RAID1/jupytertmp/mcm/src/launcher_pet2anat.sh ::: sub-s*
slicesdir -o $(for sub in sub-s*; do echo ${sub}/registration/*pet2anat_Warped.nii.gz ${sub}/ants-cortical-thickness/*desc-preproc_label-brain_T1w.nii.gz; done)
mv slicesdir pet2anat-qc
```

```bash
rsync -Pruh "rbelenya@xnat.tumnic.mgruber.eu:/RAID1/jupytertmp/mcm/data/bids/derivatives/*pet2anat-qc" ~/Downloads
```

## `func -> anat`

### BBR registration

mimic the cpac. Use this for mcm in anatomical space. Use 3mm resampled anat here istead of the original 1mm  

https://fsl.fmrib.ox.ac.uk/fsl/fslwiki/FLIRT_BBR

```bash
parallel --jobs 20 --progress /RAID1/jupytertmp/mcm/src/launcher_func2anat.sh ::: sub-s*
slicesdir -o $(for sub in sub-s*; do echo ${sub}/registration/*func2anat_bbr.nii.gz ${sub}/ants-cortical-thickness/*desc-preproc_label-brain_T1w.nii.gz; done)
mv slicesdir func2anat-qc
```

In [ ]:
for subject in subjects:
    epi = filenames.bold_mean.format(id=subject)
    t1 = filenames.t1.format(id=subject)
    fsl_mat = f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{subject:03}/registration/sub-s{subject:03}_func2anat_bbr.mat'
    itk_mat = filenames.func2anat.format(id=subject)

    ! docker exec {container} c3d_affine_tool \
        -src {epi} \
        -ref {t1} \
        {fsl_mat} \
        -fsl2ras \
        -oitk {itk_mat}

## `dwi -> anat`

```bash
parallel --jobs 20 --progress /RAID1/jupytertmp/mcm/src/launcher_dwi2anat.sh ::: sub-s*
slicesdir -o $(for sub in sub-s*; do echo ${sub}/registration/*dwi2anat_bbr.nii.gz ${sub}/ants-cortical-thickness/*desc-preproc_label-brain_T1w.nii.gz; done)
mv slicesdir dwi2anat-qc
```

In [ ]:
for subject in subjects:
    b0 = filenames.b0.format(id=subject)
    t1 = filenames.t1.format(id=subject)
    fsl_mat = f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{subject:03}/registration/sub-s{subject:03}_dwi2anat_bbr.mat'
    itk_mat = filenames.dwi2anat.format(id=subject)

    ! docker exec {container} c3d_affine_tool \
        -ref {t1} \
        -src {b0} \
        {fsl_mat} \
        -fsl2ras \
        -oitk {itk_mat}

---

# Apply transformations

## ROI Atlas

In [ ]:
assert pymcm.functions.imgs_same_space(nib.load(ATLAS), nib.load(MNI))

In [ ]:
atlas = filenames.SCHAEFER

for subject in subjects:

    space_refs = dict(
        func = filenames.bold_mean.format(id=subject),
        pet = filenames.cmrglc_3mm.format(id=subject),
        dwi = filenames.b0.format(id=subject),
    )

    anat2mni_affine = filenames.anat2mni1mm_aff.format(id=subject)
    anat2mni_invwarp = filenames.anat2mni1mm_invwarp.format(id=subject)

    for space, ref in space_refs.items():

        space2anat = filenames.get(f'{space}2anat').format(id=subject)
        out = filenames.get(f'schaefer_{space}').format(id=subject)

        ! docker exec {container} antsApplyTransforms -d 3 \
            -i {atlas} \
            -r {ref} \
            -o {out} \
            -n NearestNeighbor \
            -u short \
            -t [ {space2anat}, 1 ] \
            -t [ {anat2mni_affine}, 1 ] \
            -t {anat2mni_invwarp}

## GM mask

In [ ]:
for subject in subjects:

    gm_mask = filenames.gm_mask.format(id=subject)

    space_refs = dict(
        func = filenames.bold_mean.format(id=subject),
        pet = filenames.cmrglc_3mm.format(id=subject),
        dwi = filenames.b0.format(id=subject),
    )

    for space, ref in space_refs.items():

        space2anat = filenames.get(f'{space}2anat').format(id=subject)
        out = filenames.get(f'gm_mask_{space}').format(id=subject)

        ! docker exec {container} antsApplyTransforms -d 3 \
            -i {gm_mask} \
            -r {ref} \
            -o {out} \
            -n GenericLabel \
            -u char \
            -t [ {space2anat}, 1 ]


In [ ]:
filenames.gm_mask_func.format(id=31)

---

# Connectome

## SIFT2

In [ ]:
for subject in subjects:

    segmentation = filenames.act_5tt.format(id=subject)
    tracks = filenames.tracts.format(id=subject)
    fod = filenames.fod.format(id=subject)
    weights = filenames.sift2_weights.format(id=subject)
    mu = filenames.sift2_mu.format(id=subject)

    ! docker exec {container} tcksift2 -force -act {segmentation} -fd_scale_gm {tracks} {fod} {weights} -out_mu {mu}

## SIFT2 $W \times \mu$

In [ ]:
for subject in subjects:

    weights_file = filenames.sift2_weights.format(id=subject)
    mu_file = filenames.sift2_mu.format(id=subject)
    weightsXmu_file = filenames.sift2_weights_mu.format(id=subject)

    weights = np.loadtxt(weights_file)
    mu = np.loadtxt(mu_file)
    weightsXmu = weights * mu

    np.savetxt(weightsXmu_file, weightsXmu, fmt=r'%.10f', newline=' ')
    print(subject, end='\r')

## Tractography
### Using sifted tracts 

In [ ]:
for subject in subjects:

    tracts = filenames.tracts_sift.format(id=subject)
    # atlas = Filenames.atlas_subcort_dwi.format(sid=subject)
    atlas = filenames.schaefer_dwi.format(id=subject)
    connectome = filenames.connectome.format(id=subject)

    ! docker exec {container} tck2connectome -force -scale_invnodevol -symmetric -zero_diagonal {tracts} {atlas} {connectome}

### Using sift2 weights

In [ ]:
for subject in subjects:

    tracts = filenames.tracts.format(id=subject)
    atlas = filenames.schaefer_dwi.format(id=subject)
    weights = filenames.sift2_weights_mu.format(id=subject)
    connectome = filenames.connectome_sift2.format(id=subject)

    ! docker exec {container} tck2connectome -force -tck_weights_in {weights} -symmetric -zero_diagonal {tracts} {atlas} {connectome}

---

In [ ]:
for subject in subjects:
    connectome = pd.read_csv(filenames.connectome_sift2.format(id=subject), header=None).to_numpy()
    row = connectome[np.triu_indices_from(connectome, k=1)]
    row = row[row != 0]
    print(np.median(row))

In [ ]:
connectome = pd.read_csv(filenames.connectome.format(id=31), header=None).to_numpy()
sns.heatmap(connectome > 0, square=True, cmap='cet_CET_L1_r', norm=colors.PowerNorm(gamma=0.5))

In [ ]:
connectome2 = pd.read_csv(filenames.connectome_sift2.format(id=31), header=None).to_numpy()
sns.heatmap(connectome2 > 0, square=True, cmap='cet_CET_L1_r', norm=colors.PowerNorm(gamma=0.5))

In [ ]:
sns.kdeplot(connectome[connectome != 0])

In [ ]:
sns.kdeplot(connectome2[connectome2 != 0])

In [ ]:
q

# Degree Centrality

In [ ]:
for subject in subjects:

    gm_mask = filenames.gm_mask_func.format(id=subject)
    snr_mask = filenames.bold_snr_mask.format(id=subject)
    out = filenames.bold_snr_gm_mask.format(id=subject)

    ! docker exec {container} fslmaths {gm_mask} -mul {snr_mask} {out}

In [ ]:
threshold = 0.177734

for subject in subjects:
    
    bold = filenames.bold.format(id=subject)
    mask = filenames.bold_snr_gm_mask.format(id=subject)
    
    out = filenames.dc.format(id=subject)
    out_binary = filenames.dc_binary.format(id=subject)
    out_weighted = filenames.dc_weighted.format(id=subject)
    # out1d = f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{subject:03}/degree-centrality/sub-s{subject:03}_degree.1D'
    
    if os.path.exists(out):
        ! rm {out} {out1d}

    ! docker exec {container} 3dDegreeCentrality -thresh {threshold} -polort -1 -mask {mask} -prefix {out} {bold}
    ! docker exec {container} 3dbucket -prefix {out_binary} {out}'[0]'
    ! docker exec {container} 3dbucket -prefix {out_weighted} {out}'[1]'

## Smooth DC weighted image

In [ ]:
sigma = 6/2.355

for subject in subjects:
    
    dc = filenames.dc_weighted.format(id=subject)
    gm_mask = filenames.bold_snr_gm_mask.format(id=subject)
    dc_smooth = filenames.dc_smooth.format(id=subject)
    
    dc_med = ! docker exec {container} fslstats {dc} -k {gm_mask} -p 50
    dc_med = float(dc_med[1])

    ! docker exec {container} susan {dc} {dc_med} {sigma} 3 1 1 {gm_mask} {dc_med} {dc_smooth}
    # ! docker exec {container} fslmaths {dc_smooth} -mas {gm_mask} {dc_smooth}

---